# D7 Demographic Analysis: Victim Composition by Origin Country

This notebook extends the D7 network analysis to characterize the demographic profile of
sex-trafficking victims by country of origin, using the CTDC Global Synthetic Dataset. It
reports, for each Southeast Asian origin country, the share of victims recorded as female
and the share recorded as minors, computed only over records with a known value for each field.

In [1]:
import pandas as pd

df = pd.read_csv("data/ctdc_global_synthetic_data_v2026.csv", low_memory=False)
print("Records:", df.shape[0])
print("Columns:", df.shape[1])

Records: 268515
Columns: 27


## Step 1: Restrict to sex-trafficking records

The analysis concerns sexual exploitation, so the dataset is filtered to records flagged
`isSexualExploit == 1`, matching the scope of the network analysis.

In [2]:
# Keep only sex-trafficking records
sex_df = df[df["isSexualExploit"] == 1]
print("Sex-trafficking records:", len(sex_df))

Sex-trafficking records: 100390


## Step 2: Isolate one origin country (Vietnam)

To develop and verify the demographic method on a single case before generalizing, records
are filtered to Vietnam-origin victims (`citizenship == "VNM"`).

In [3]:
# Vietnam-origin sex-trafficking records
vnm = sex_df[sex_df["citizenship"] == "VNM"]
print("Vietnam-origin sex records:", len(vnm))

Vietnam-origin sex records: 1070


### Gender composition (Vietnam)

The share of female victims is computed over records with a recorded gender (missing values
excluded from the denominator), and the proportion of records with gender unrecorded is
reported alongside it.

In [4]:
# Vietnam gender composition (denominator = records with a KNOWN gender)
gender_known = vnm["gender"].notna().sum()
n_women      = (vnm["gender"] == "Woman").sum()
pct_female   = round(100 * n_women / gender_known, 1)
pct_missing  = round(100 * vnm["gender"].isna().sum() / len(vnm), 1)

print("Records with known gender:", gender_known)
print("Women:", n_women)
print("Percent female (of known gender):", pct_female)
print("Gender missing:", pct_missing, "%")

Records with known gender: 886
Women: 886
Percent female (of known gender): 100.0
Gender missing: 17.2 %


### Age composition (Vietnam)

The share of minors (victims recorded in the 0-8 or 9-17 age bands) is computed over records
with a recorded age, with the proportion of records missing age reported alongside.

In [5]:
# Vietnam age composition (denominator = records with a KNOWN age)
minor_bands = ["0--8", "09--17"]
age_known   = vnm["ageBroad"].notna().sum()
n_minors    = vnm["ageBroad"].isin(minor_bands).sum()
pct_minor   = round(100 * n_minors / age_known, 1)
pct_missing = round(100 * vnm["ageBroad"].isna().sum() / len(vnm), 1)

print("Records with known age:", age_known)
print("Minors (under 18):", n_minors)
print("Percent minor (of known age):", pct_minor)
print("Age missing:", pct_missing, "%")

Records with known age: 650
Minors (under 18): 191
Percent minor (of known age): 29.4
Age missing: 39.3 %


### Demographic composition by origin country (all five)

Gender and age composition is computed for each origin country using the same procedure:
proportions are taken over records with a recorded value, and the missing-data rate is
reported for each field.

In [6]:
countries = {"KHM": "Cambodia", "THA": "Thailand", "VNM": "Vietnam",
             "IDN": "Indonesia", "PHL": "Philippines"}
minor_bands = ["0--8", "09--17"]

rows = []
for code, name in countries.items():
    sub = sex_df[sex_df["citizenship"] == code]
    n = len(sub)

    gender_known = sub["gender"].notna().sum()
    pct_female   = round(100 * (sub["gender"] == "Woman").sum() / gender_known, 1)
    gender_miss  = round(100 * sub["gender"].isna().sum() / n, 1)

    age_known = sub["ageBroad"].notna().sum()
    pct_minor = round(100 * sub["ageBroad"].isin(minor_bands).sum() / age_known, 1)
    age_miss  = round(100 * sub["ageBroad"].isna().sum() / n, 1)

    rows.append({
        "country": name, "records": n,
        "gender_known": gender_known, "pct_female": pct_female, "gender_miss%": gender_miss,
        "age_known": age_known, "pct_minor": pct_minor, "age_miss%": age_miss,
    })

demographics = pd.DataFrame(rows)
print(demographics.to_string(index=False))

    country  records  gender_known  pct_female  gender_miss%  age_known  pct_minor  age_miss%
   Cambodia      960           843        55.3          12.2        586       57.7       39.0
   Thailand      300           213        95.8          29.0        145       13.8       51.7
    Vietnam     1070           886       100.0          17.2        650       29.4       39.3
  Indonesia      317           224       100.0          29.3         86       33.7       72.9
Philippines      977           821        97.1          16.0        146       36.3       85.1


### Save demographic results

The demographic composition table is written to the outputs folder for use as an appendix and
as country-specific evidence in the Social evaluation row.

In [7]:
import os
os.makedirs("outputs", exist_ok=True)
demographics.to_csv("outputs/GEOG594A_D7_DemographicsByOrigin.csv", index=False)
print("Saved: outputs/GEOG594A_D7_DemographicsByOrigin.csv")

Saved: outputs/GEOG594A_D7_DemographicsByOrigin.csv
